# Supply Chain Analytics — Step by Step
This notebook walks through everything: generating the data, cleaning it, exploring it, building features, and running optimization models.

**How to use this notebook:**
- Run each cell in order, from top to bottom (click the cell, press `Shift + Enter`)
- Read the explanation above each code cell BEFORE running it
- If a cell gives an error, just tell Claude what the error says


## Step 0: Load the Tools (Libraries)

Before we do anything, Python needs to load some "toolboxes":
- **pandas** → for working with tables of data (like Excel, but in code)
- **numpy** → for math and number operations
- **random / datetime** → to generate fake dates and random numbers

Run this cell first.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## Step 1: Create the Dataset

We're going to create a **fake but realistic** supply chain dataset for a Nigerian company with:
- 8 products (SKUs) — like Rice, Cooking Oil, Hand Sanitizer, etc.
- 5 suppliers — some local (fast, reliable), some international (slow, less reliable)
- 4 warehouses — Lagos, Abuja, Port Harcourt, Kano
- 2,000 orders spread across 2023–2024

This cell sets up the "reference information" — the list of suppliers, warehouses, and products, and which suppliers can supply which products.

**Just run it — nothing will print yet, we're just storing information.**

In [2]:
# Reference data: suppliers, warehouses, products

suppliers = {
    "SUP-001": {"name": "AgroFresh Ltd",        "country": "Nigeria",  "reliability": 0.92, "base_lead": 5},
    "SUP-002": {"name": "PanAfrica Supplies",   "country": "Ghana",    "reliability": 0.85, "base_lead": 8},
    "SUP-003": {"name": "MedTech Distributors", "country": "India",    "reliability": 0.78, "base_lead": 14},
    "SUP-004": {"name": "EastTrade Corp",       "country": "China",    "reliability": 0.80, "base_lead": 21},
    "SUP-005": {"name": "LocalMart Wholesale",  "country": "Nigeria",  "reliability": 0.95, "base_lead": 3},
}

warehouses = {
    "WH-LOS": {"city": "Lagos",        "capacity": 50000},
    "WH-ABJ": {"city": "Abuja",        "capacity": 30000},
    "WH-PHC": {"city": "Port Harcourt","capacity": 25000},
    "WH-KAN": {"city": "Kano",         "capacity": 20000},
}

products = {
    "SKU-A001": {"name": "Rice (50kg bag)",       "category": "Food",        "unit_cost": 18000, "reorder_pt": 200, "reorder_qty": 500},
    "SKU-A002": {"name": "Cooking Oil (5L)",      "category": "Food",        "unit_cost": 6500,  "reorder_pt": 150, "reorder_qty": 400},
    "SKU-B001": {"name": "Hand Sanitizer (1L)",   "category": "Healthcare",  "unit_cost": 1800,  "reorder_pt": 300, "reorder_qty": 800},
    "SKU-B002": {"name": "Paracetamol (100tabs)", "category": "Healthcare",  "unit_cost": 850,   "reorder_pt": 500, "reorder_qty": 1000},
    "SKU-C001": {"name": "A4 Paper (500 sheets)", "category": "Stationery", "unit_cost": 3500,  "reorder_pt": 100, "reorder_qty": 300},
    "SKU-C002": {"name": "Ballpoint Pens (Box)",  "category": "Stationery", "unit_cost": 1200,  "reorder_pt": 200, "reorder_qty": 500},
    "SKU-D001": {"name": "Cement (50kg bag)",     "category": "Construction","unit_cost": 5500, "reorder_pt": 300, "reorder_qty": 600},
    "SKU-D002": {"name": "Paint (20L bucket)",    "category": "Construction","unit_cost": 12000,"reorder_pt": 80,  "reorder_qty": 200},
}

# Which suppliers can supply which products
sku_supplier_map = {
    "SKU-A001": ["SUP-001","SUP-005"], "SKU-A002": ["SUP-001","SUP-002"],
    "SKU-B001": ["SUP-003","SUP-005"], "SKU-B002": ["SUP-003","SUP-004"],
    "SKU-C001": ["SUP-004","SUP-005"], "SKU-C002": ["SUP-002","SUP-005"],
    "SKU-D001": ["SUP-001","SUP-002"], "SKU-D002": ["SUP-003","SUP-004"],
}

print("✅ Reference data ready!")

✅ Reference data ready!


## Step 2: Generate 2,000 Orders

Now we'll create 2,000 rows of order data. For each order, we randomly pick:
- A product (SKU)
- A warehouse
- A supplier (from the ones allowed for that product)
- A random date between Jan 2023 and Dec 2024
- A quantity (with some months — May to July — having higher demand, simulating seasonality)
- Costs, lead times, and whether the delivery was delayed

**This is the "data generation" step — it builds your dataset from scratch.**

This might take a few seconds to run.

In [3]:
start_date = datetime(2023, 1, 1)
end_date   = datetime(2024, 12, 31)
days_range = (end_date - start_date).days

rows = []
for i in range(2000):
    order_date = start_date + timedelta(days=random.randint(0, days_range))
    sku        = random.choice(list(products.keys()))
    wh         = random.choice(list(warehouses.keys()))
    sup_id     = random.choice(sku_supplier_map[sku])
    sup        = suppliers[sup_id]
    prod       = products[sku]

    # Seasonal effect: demand is higher around May-July
    month      = order_date.month
    season_fx  = 1 + 0.3 * np.sin((month - 3) * np.pi / 6)
    qty        = max(1, int(np.random.poisson(50 * season_fx)))

    # Lead time: how long delivery actually took (with possible delays)
    base_lead  = sup["base_lead"]
    delay      = 0 if random.random() < sup["reliability"] else random.randint(1, 10)
    actual_lead= max(1, base_lead + delay + random.randint(-1, 2))

    # Costs
    unit_cost  = prod["unit_cost"] * np.random.uniform(0.92, 1.12)
    ship_cost  = round(qty * random.uniform(80, 300), 2)

    # Inventory levels
    inv_before = random.randint(50, 800)
    inv_after  = max(0, inv_before - qty + random.randint(0, 200))

    rows.append({
        "order_id":          f"ORD-{10000+i}",
        "order_date":        order_date.strftime("%Y-%m-%d"),
        "sku":               sku,
        "product_name":      prod["name"],
        "category":          prod["category"],
        "warehouse_id":      wh,
        "warehouse_city":    warehouses[wh]["city"],
        "supplier_id":       sup_id,
        "supplier_name":     sup["name"],
        "supplier_country":  sup["country"],
        "order_quantity":    qty,
        "unit_cost_ngn":     round(unit_cost, 2),
        "total_order_cost":  round(unit_cost * qty, 2),
        "shipping_cost":     ship_cost,
        "total_cost":        round(unit_cost * qty + ship_cost, 2),
        "promised_lead_days":base_lead,
        "actual_lead_days":  actual_lead,
        "delay_days":        max(0, actual_lead - base_lead),
        "delivery_delayed":  1 if actual_lead > base_lead else 0,
        "inventory_before":  inv_before,
        "inventory_after":   inv_after,
        "reorder_point":     prod["reorder_pt"],
        "stockout_flag":     1 if inv_before < qty else 0,
        "supplier_reliability_score": sup["reliability"],
    })

df = pd.DataFrame(rows)
df["order_date"] = pd.to_datetime(df["order_date"])

print(f"✅ Created {len(df)} orders!")
df.head()

✅ Created 2000 orders!


,order_id,order_date,sku,product_name,category,warehouse_id,warehouse_city,supplier_id,supplier_name,supplier_country,...,total_cost,promised_lead_days,actual_lead_days,delay_days,delivery_delayed,inventory_before,inventory_after,reorder_point,stockout_flag,supplier_reliability_score
0,ORD-10000,2024-10-16,SKU-A002,Cooking Oil (5L),Food,WH-LOS,Lagos,SUP-002,PanAfrica Supplies,Ghana,...,286944.63,8,8,0,0,742,891,150,0,0.85
1,ORD-10001,2024-07-12,SKU-A002,Cooking Oil (5L),Food,WH-KAN,Kano,SUP-001,AgroFresh Ltd,Nigeria,...,410408.59,5,5,0,0,666,607,150,0,0.92
2,ORD-10002,2024-07-28,SKU-B002,Paracetamol (100tabs),Healthcare,WH-KAN,Kano,SUP-003,MedTech Distributors,India,...,67832.32,14,15,1,1,56,185,500,1,0.78
3,ORD-10003,2023-06-13,SKU-D001,Cement (50kg bag),Construction,WH-PHC,Port Harcourt,SUP-002,PanAfrica Supplies,Ghana,...,311125.41,8,9,1,1,439,405,300,0,0.85
4,ORD-10004,2024-01-03,SKU-C002,Ballpoint Pens (Box),Stationery,WH-PHC,Port Harcourt,SUP-002,PanAfrica Supplies,Ghana,...,49557.82,8,7,0,0,437,424,200,0,0.85


## Step 3: Save the Dataset

This saves your dataset as a CSV file in the same folder as this notebook, so you can open it in Excel later if you want.

In [4]:
df.to_csv("supply_chain_data.csv", index=False)
print("✅ Saved as supply_chain_data.csv")

✅ Saved as supply_chain_data.csv


## Step 4: Look at the Data — Quick Overview

Let's check basic facts about our dataset:
- How many rows and columns?
- What date range?
- How much money was spent in total?
- How many suppliers, warehouses, products?

This is just to confirm everything looks right.

In [5]:
print(f"Number of orders: {len(df):,}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print(f"Number of unique products (SKUs): {df['sku'].nunique()}")
print(f"Number of warehouses: {df['warehouse_id'].nunique()}")
print(f"Number of suppliers: {df['supplier_id'].nunique()}")
print(f"Total amount spent: ₦{df['total_cost'].sum():,.0f}")

Number of orders: 2,000
Date range: 2023-01-01 to 2024-12-31
Number of unique products (SKUs): 8
Number of warehouses: 4
Number of suppliers: 5
Total amount spent: ₦641,501,881


## Step 5: Data Cleaning

Real datasets often have problems:
- Duplicate orders (same order entered twice)
- Negative or zero quantities/costs (data entry errors)
- Crazy lead times (like -5 days, or 500 days)
- Extreme outliers in order quantity

This cell checks for and fixes these issues. **It's a safety check** — even if our data is already clean, this makes sure.

In [6]:
original_count = len(df)

# 1. Remove duplicate orders (same order_id appearing more than once)
df = df.drop_duplicates(subset="order_id")

# 2. Remove rows where quantity or cost is zero or negative (impossible in real life)
for col in ["order_quantity", "unit_cost_ngn", "total_cost", "shipping_cost"]:
    df = df[df[col] > 0]

# 3. Cap lead times to a realistic range (1 to 180 days)
df["actual_lead_days"] = df["actual_lead_days"].clip(1, 180)

# 4. Remove extreme outliers in order quantity
#    (anything way higher or lower than the typical range)
Q1 = df["order_quantity"].quantile(0.25)
Q3 = df["order_quantity"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR
df = df[(df["order_quantity"] >= lower_bound) & (df["order_quantity"] <= upper_bound)]

# 5. Standardize SKU codes (uppercase, no extra spaces)
df["sku"] = df["sku"].str.upper().str.strip()

df = df.reset_index(drop=True)

print(f"✅ Cleaning done. Started with {original_count} rows, now have {len(df)} rows.")

✅ Cleaning done. Started with 2000 rows, now have 2000 rows.


## Step 6: Feature Engineering — Creating New Useful Columns

"Feature engineering" just means: **calculating new columns from existing data** that are more useful for analysis.

We'll create:

1. **Rolling 30-day & 90-day average demand** — the average order quantity over the last 30/90 days for each product. This smooths out day-to-day randomness and shows the real trend.

2. **Lead time variability** — how consistent (or unpredictable) each supplier's delivery times are. A supplier with high variability is risky even if their average is okay.

3. **Composite supplier score** — a single score (0 to 1) combining reliability, on-time rate, and delay severity. Higher = better supplier.

4. **Inventory turnover proxy** — roughly, how much of the available inventory was used in each order. Higher = inventory moves fast (good); lower = inventory sits around (ties up cash).

5. **Cost per unit** — total cost divided by quantity, useful for comparing "true" cost across different order sizes.

6. **Peak season flag** — marks orders placed in May, June, or July (our high-demand months) as 1, others as 0.

In [7]:
# Sort by date first (important for rolling calculations)
df = df.sort_values("order_date").reset_index(drop=True)

# 1. Rolling average demand (30-day and 90-day) per product
df["rolling_30d_demand"] = (
    df.groupby("sku")["order_quantity"]
    .transform(lambda x: x.rolling(30, min_periods=1).mean())
    .round(1)
)
df["rolling_90d_demand"] = (
    df.groupby("sku")["order_quantity"]
    .transform(lambda x: x.rolling(90, min_periods=1).mean())
    .round(1)
)

# 2. Lead time variability per supplier (standard deviation of lead times)
lead_time_std = df.groupby("supplier_id")["actual_lead_days"].std().rename("lead_time_std")
df = df.merge(lead_time_std, on="supplier_id", how="left")

# 3. Composite supplier score (0 = bad, 1 = great)
df["composite_sup_score"] = (
    df["supplier_reliability_score"] * 0.50 +
    (1 - df["delivery_delayed"])      * 0.30 +
    (1 / (1 + df["delay_days"]))      * 0.20
).round(3)

# 4. Inventory turnover proxy (how much inventory was used per order)
df["inv_turnover_proxy"] = (
    df["order_quantity"] / df["inventory_before"].replace(0, np.nan)
).round(3)

# 5. Cost per unit
df["total_cost_per_unit"] = (df["total_cost"] / df["order_quantity"]).round(2)

# 6. Peak season flag (May, June, July = 1)
df["month"] = df["order_date"].dt.month
df["year"]  = df["order_date"].dt.year
df["is_peak_season"] = df["month"].isin([5, 6, 7]).astype(int)

print("✅ New columns created!")
df[["sku", "rolling_30d_demand", "rolling_90d_demand", "lead_time_std",
    "composite_sup_score", "inv_turnover_proxy", "total_cost_per_unit", "is_peak_season"]].head()

✅ New columns created!


,sku,rolling_30d_demand,rolling_90d_demand,lead_time_std,composite_sup_score,inv_turnover_proxy,total_cost_per_unit,is_peak_season
0,SKU-C002,33.0,33.0,1.831334,0.975,0.050,1349.11,0
1,SKU-C001,46.0,46.0,2.612122,0.422,0.062,4100.35,0
2,SKU-B002,28.0,28.0,2.580205,0.890,0.082,987.51,0
3,SKU-D001,29.0,29.0,2.480676,0.925,0.095,6011.56,0
4,SKU-C002,37.0,37.0,1.831334,0.575,0.075,1406.30,0


## Step 7: Explore Demand Patterns

Let's see, for each product, how much the demand varies. We calculate the **Coefficient of Variation (CV)**:

> CV = standard deviation ÷ average

A higher CV means the demand for that product swings around a lot (unpredictable). A lower CV means demand is steady and easier to plan for.

In [8]:
demand_summary = df.groupby("sku")["order_quantity"].agg(
    average_qty="mean",
    std_dev="std"
)
demand_summary["variability_score"] = (demand_summary["std_dev"] / demand_summary["average_qty"]).round(3)

print("Demand Variability by Product (sorted from most to least unpredictable):")
demand_summary.sort_values("variability_score", ascending=False)

Demand Variability by Product (sorted from most to least unpredictable):


,average_qty,std_dev,variability_score
sku,,,
SKU-D001,49.507874,13.045600,0.264
SKU-B002,50.375000,13.173419,0.262
SKU-C002,48.961538,12.804012,0.262
SKU-C001,50.675781,13.241363,0.261
SKU-A002,49.574713,12.528397,0.253
SKU-B001,51.022556,12.792077,0.251
SKU-A001,50.075314,12.491786,0.249
SKU-D002,51.128099,12.751215,0.249


## Step 8: Explore Supplier Performance

Now let's see how each supplier is doing:
- **delay_rate**: what % of their orders were late
- **avg_delay**: on average, how many days late (when they were late)
- **composite_score**: the overall score we calculated earlier (higher = better)

This tells us at a glance who our best and worst suppliers are.

In [9]:
supplier_summary = df.groupby(["supplier_id", "supplier_name"]).agg(
    delay_rate=("delivery_delayed", "mean"),
    avg_delay=("delay_days", "mean"),
    composite_score=("composite_sup_score", "mean"),
).round(3)

supplier_summary["delay_rate"] = (supplier_summary["delay_rate"] * 100).round(1)  # convert to %

print("Supplier Performance (sorted from best to worst):")
supplier_summary.sort_values("composite_score", ascending=False)

Supplier Performance (sorted from best to worst):


,,delay_rate,avg_delay,composite_score
supplier_id,supplier_name,,,
SUP-005,LocalMart Wholesale,51.7,1.031,0.757
SUP-001,AgroFresh Ltd,55.7,1.368,0.723
SUP-002,PanAfrica Supplies,55.7,1.445,0.687
SUP-004,EastTrade Corp,58.0,1.646,0.650
SUP-003,MedTech Distributors,61.2,1.722,0.626


## Step 9: Optimization Model 1 — EOQ (Economic Order Quantity)

**What is EOQ?**

EOQ answers the question: *"What is the ideal quantity to order each time, to minimize total cost?"*

There's a tradeoff:
- Order **too much** at once → you pay more to store it (holding cost)
- Order **too little, too often** → you pay more in ordering fees (shipping, admin, etc.)

EOQ finds the "sweet spot" using this formula:

> EOQ = √(2 × Annual Demand × Ordering Cost ÷ Holding Cost per Unit)

We assume:
- Ordering cost = ₦5,000 per order (admin/shipping overhead)
- Holding cost = 20% of the product's unit cost per year (storage, insurance, etc.)

The result tells you: **"order this many units, this often."**

In [10]:
ordering_cost = 5000   # NGN per order
holding_rate = 0.20    # 20% of unit cost per year

eoq_table = df.groupby("sku").agg(
    total_demand=("order_quantity", "sum"),
    avg_unit_cost=("unit_cost_ngn", "mean")
)

# Our data covers 2 years, so divide by 2 to get yearly demand
eoq_table["annual_demand"] = eoq_table["total_demand"] / 2
eoq_table["holding_cost_per_unit"] = eoq_table["avg_unit_cost"] * holding_rate

# EOQ formula
eoq_table["EOQ"] = np.sqrt(
    2 * eoq_table["annual_demand"] * ordering_cost / eoq_table["holding_cost_per_unit"]
).round(0)

# How often to order (times per year, and days between orders)
eoq_table["orders_per_year"] = (eoq_table["annual_demand"] / eoq_table["EOQ"]).round(1)
eoq_table["reorder_every_X_days"] = (365 / eoq_table["orders_per_year"]).round(0)

print("Recommended Order Quantities (EOQ):")
eoq_table[["annual_demand", "EOQ", "orders_per_year", "reorder_every_X_days"]]

Recommended Order Quantities (EOQ):


,annual_demand,EOQ,orders_per_year,reorder_every_X_days
sku,,,,
SKU-A001,5984.0,128.0,46.8,8.0
SKU-A002,6469.5,221.0,29.3,12.0
SKU-B001,6786.0,431.0,15.7,23.0
SKU-B002,6246.5,601.0,10.4,35.0
SKU-C001,6486.5,302.0,21.5,17.0
SKU-C002,5728.5,484.0,11.8,31.0
SKU-D001,6287.5,237.0,26.5,14.0
SKU-D002,6186.5,159.0,38.9,9.0


## Step 10: Optimization Model 2 — Safety Stock & Reorder Point

**What is Safety Stock?**

It's extra inventory you keep "just in case" — to protect against:
- Demand suddenly spiking
- Supplier deliveries arriving late

**What is Reorder Point (ROP)?**

It's the inventory level at which you should place a new order. If your stock drops to this level, order now — by the time the new stock arrives, you'll be at (or just above) zero.

> ROP = (Average Demand × Average Lead Time) + Safety Stock

We're calculating this for a **95% service level** — meaning we'll have enough stock 95% of the time (industry-standard target).

In [11]:
z_score = 1.65  # this number corresponds to a 95% service level

safety_stock_table = df.groupby(["sku", "supplier_id"]).agg(
    avg_lead_time=("actual_lead_days", "mean"),
    std_lead_time=("actual_lead_days", "std"),
    avg_demand=("order_quantity", "mean"),
    std_demand=("order_quantity", "std"),
).reset_index()

# Safety stock formula (accounts for both demand AND lead time variability)
safety_stock_table["safety_stock"] = (
    z_score * np.sqrt(
        safety_stock_table["avg_lead_time"] * safety_stock_table["std_demand"]**2 +
        safety_stock_table["avg_demand"]**2 * safety_stock_table["std_lead_time"].fillna(0)**2
    )
).round(0)

# Reorder point
safety_stock_table["reorder_point"] = (
    safety_stock_table["avg_demand"] * safety_stock_table["avg_lead_time"] + safety_stock_table["safety_stock"]
).round(0)

print("Recommended Safety Stock & Reorder Points (95% service level):")
safety_stock_table[["sku", "supplier_id", "avg_lead_time", "safety_stock", "reorder_point"]]

Recommended Safety Stock & Reorder Points (95% service level):


,sku,supplier_id,avg_lead_time,safety_stock,reorder_point
0,SKU-A001,SUP-001,5.973451,165.0,466.0
1,SKU-A001,SUP-005,3.944444,162.0,358.0
2,SKU-A002,SUP-001,6.237288,189.0,500.0
3,SKU-A002,SUP-002,9.454545,219.0,686.0
4,SKU-B001,SUP-003,15.536232,215.0,995.0
5,SKU-B001,SUP-005,3.757812,174.0,369.0
6,SKU-B002,SUP-003,15.311321,221.0,1006.0
7,SKU-B002,SUP-004,22.345070,232.0,1342.0
8,SKU-C001,SUP-004,22.535433,254.0,1410.0
9,SKU-C001,SUP-005,3.782946,157.0,346.0


## Step 11: Optimization Model 3 — Supplier Scoring (Who to Prioritize)

We already calculated a basic score per order. Now let's build a **final ranking** of all suppliers using 4 factors:

| Factor | Weight | Meaning |
|---|---|---|
| On-time rate | 35% | How often they deliver on time |
| Average delay | 25% | When late, how late (lower = better) |
| Lead time consistency | 20% | How predictable their delivery times are |
| Unit cost | 20% | How expensive their products are (lower = better) |

Each factor is converted to a 0–1 scale, then combined using the weights above. The result is one final number per supplier — **higher = better overall**.

In [12]:
supplier_scores = df.groupby(["supplier_id", "supplier_name"]).agg(
    on_time_rate=("delivery_delayed", lambda x: 1 - x.mean()),
    avg_delay=("delay_days", "mean"),
    lead_variability=("actual_lead_days", "std"),
    avg_unit_cost=("unit_cost_ngn", "mean"),
).reset_index()

# Helper function: scales any column to a 0-1 range
def normalize(column, lower_is_better=False):
    scaled = (column - column.min()) / (column.max() - column.min())
    return 1 - scaled if lower_is_better else scaled

supplier_scores["score_on_time"]    = normalize(supplier_scores["on_time_rate"])
supplier_scores["score_delay"]      = normalize(supplier_scores["avg_delay"], lower_is_better=True)
supplier_scores["score_consistency"]= normalize(supplier_scores["lead_variability"], lower_is_better=True)
supplier_scores["score_cost"]       = normalize(supplier_scores["avg_unit_cost"], lower_is_better=True)

# Combine using weights
supplier_scores["final_score"] = (
    supplier_scores["score_on_time"]     * 0.35 +
    supplier_scores["score_delay"]       * 0.25 +
    supplier_scores["score_consistency"] * 0.20 +
    supplier_scores["score_cost"]        * 0.20
).round(3)

print("Final Supplier Ranking (best to worst):")
supplier_scores[["supplier_id", "supplier_name", "on_time_rate", "avg_delay", "final_score"]].sort_values(
    "final_score", ascending=False
)

Final Supplier Ranking (best to worst):


,supplier_id,supplier_name,on_time_rate,avg_delay,final_score
4,SUP-005,LocalMart Wholesale,0.483366,1.031311,0.946
1,SUP-002,PanAfrica Supplies,0.442667,1.445333,0.535
0,SUP-001,AgroFresh Ltd,0.442897,1.367688,0.405
3,SUP-004,EastTrade Corp,0.419948,1.645669,0.332
2,SUP-003,MedTech Distributors,0.387701,1.721925,0.190


## Step 12: Summary — What This All Means

Run this last cell to print a plain-English summary of the key takeaways from your data.

In [13]:
delay_rate = df['delivery_delayed'].mean() * 100
worst_supplier = supplier_scores.sort_values("final_score").iloc[0]
best_supplier = supplier_scores.sort_values("final_score", ascending=False).iloc[0]
highest_stockout_sku = df.groupby("sku")["stockout_flag"].mean().idxmax()
highest_stockout_rate = df.groupby("sku")["stockout_flag"].mean().max() * 100

print("="*60)
print("KEY TAKEAWAYS")
print("="*60)
print(f"\n1. {delay_rate:.1f}% of all deliveries were late.")
print(f"   This is the biggest issue to fix first.")
print(f"\n2. Best supplier: {best_supplier['supplier_name']} (score: {best_supplier['final_score']})")
print(f"   Worst supplier: {worst_supplier['supplier_name']} (score: {worst_supplier['final_score']})")
print(f"   Consider shifting more orders to your best supplier.")
print(f"\n3. Product with most stockouts: {highest_stockout_sku} ({highest_stockout_rate:.1f}% of orders)")
print(f"   This product needs a higher safety stock / reorder point.")
print(f"\n4. Use the EOQ table (Step 9) to know exactly how much to order each time.")
print(f"\n5. Use the Safety Stock table (Step 10) to know when to reorder.")
print("\n" + "="*60)

KEY TAKEAWAYS

1. 56.1% of all deliveries were late.
   This is the biggest issue to fix first.

2. Best supplier: LocalMart Wholesale (score: 0.946)
   Worst supplier: MedTech Distributors (score: 0.19)
   Consider shifting more orders to your best supplier.

3. Product with most stockouts: SKU-A001 (1.3% of orders)
   This product needs a higher safety stock / reorder point.

4. Use the EOQ table (Step 9) to know exactly how much to order each time.

5. Use the Safety Stock table (Step 10) to know when to reorder.

